# Direct neural-network evaluation on 2019
Train ten seeded networks on all elections through 2015. Use the whole 2017
election for early stopping, retaining each network's best checkpoint. Evaluate
their averaged probabilities on 2019. No 2019 rows are used for training or
checkpoint selection. The 2024 test CSV is loaded separately and remains unused.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "Models" / "NN01_model.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from Models import NN01_model as nn

train = pd.read_csv(project_root / "TEST_TRAIN" / "train.csv")
test = pd.read_csv(project_root / "TEST_TRAIN" / "test.csv")  # 2024: unused here.
years = pd.to_numeric(train["election"], errors="raise")
training_data = train.loc[years <= 2015].copy()
validation_data = train.loc[years == 2017].copy()
evaluation_data = train.loc[years == 2019].dropna(subset=["winner"]).copy()
for name, frame in [("Training through 2015", training_data),
                    ("2017 early stopping", validation_data),
                    ("2019 evaluation", evaluation_data)]:
    if frame.empty or frame["winner"].isna().any():
        raise ValueError(f"{name} requires nonempty data with known winners.")
    print(f"{name}: {len(frame)} rows")


In [ ]:
# Use the existing network trainer with explicit election splits.
# model.train() would instead split the latest election in half.
model = nn.NeuralNetworkModel()
model.label_encoder = LabelEncoder().fit(training_data["winner"])
model.classes_ = model.label_encoder.classes_
unknown = set(validation_data["winner"]) - set(model.classes_)
if unknown:
    raise ValueError(f"2017 contains parties absent from training: {unknown}")

for seed in nn.SEEDS:
    network, preprocessor, record = nn._train_network(
        training_data, validation_data, model.label_encoder, seed, nn.MAX_EPOCHS
    )
    model.networks.append(network)
    model.preprocessors.append(preprocessor)
    model.final_records.append(record)
    print(f"Seed {seed}: best epoch {record['best_epoch']}, "
          f"stopped at {record['stopping_epoch']}")
display(pd.DataFrame(model.final_records))


## 2019 evaluation
Report each network's accuracy and the ensemble accuracy on all 2019 rows with
known winners. Missing predictors use the fitted training-data imputers. Also
report ensemble accuracy on complete rows to match the pipeline's comparison subset.


In [ ]:
if len(model.networks) != len(nn.SEEDS):
    raise ValueError("Finish training all ten networks before evaluation.")
run_scores = []
for seed, network, preprocessor in zip(nn.SEEDS, model.networks, model.preprocessors):
    features = nn.as_features(preprocessor.transform(evaluation_data[nn.FEATURE_COLUMNS]))
    network.eval()
    with torch.inference_mode():
        encoded = network(features).argmax(dim=1).numpy()
    predictions = model.label_encoder.inverse_transform(encoded)
    run_scores.append({"seed": seed, "2019_accuracy": accuracy_score(
        evaluation_data["winner"], predictions
    )})
display(pd.DataFrame(run_scores))

# predict() averages party probabilities across all ten retained checkpoints.
evaluation_predictions = model.predict(evaluation_data)
evaluation_accuracy = accuracy_score(evaluation_data["winner"], evaluation_predictions)
print(f"2019 test accuracy (10-network ensemble): {evaluation_accuracy:.2%}")
print(f"Evaluated {len(evaluation_data)} rows")
complete = evaluation_data[nn.FEATURE_COLUMNS].notna().all(axis=1)
if complete.any():
    complete_accuracy = accuracy_score(
        evaluation_data.loc[complete, "winner"], evaluation_predictions[complete.to_numpy()]
    )
    print(f"2019 complete-row accuracy: {complete_accuracy:.2%} ({complete.sum()} rows)")
